# KNN imputer: filling Titanic `Age` from similar passengers

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
The data (`data/titanic_knn.csv`) is four columns of the Kaggle Titanic `train.csv` (891 passengers): `Age`, `Pclass`, `Fare` and the target `Survived`.

## 1. The worked example: nan-Euclidean distance by hand and by scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import nan_euclidean_distances

# the 5-row table of the Note; row 2 (index 1) has a gap in f1
toy = np.array([[30, 60, np.nan],
                [np.nan, 55, 20],
                [40, 52, 25],
                [25, np.nan, 22],
                [50, 70, 40]])

# distance from row 2 to every row (0 to itself)
nan_euclidean_distances(toy[[1]], toy).round(2)

In [ ]:
# the same distance by hand, for row 2 and row 3
a, b = toy[1], toy[2]
both = ~np.isnan(a) & ~np.isnan(b)                # columns where both rows have a value
weight = len(a) / both.sum()                      # 3 columns / 2 used = 1.5
np.sqrt(weight * ((a[both] - b[both]) ** 2).sum())   # 7.14

In [ ]:
from sklearn.impute import KNNImputer

# fill the gap from the 2 nearest rows: plain mean, then distance-weighted
print(KNNImputer(n_neighbors=2).fit_transform(toy)[1])
print(KNNImputer(n_neighbors=2, weights="distance").fit_transform(toy)[1])

## 2. Load the Titanic data and split it

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/titanic_knn.csv")
df.isnull().mean() * 100          # share of missing values per column: only Age has gaps

In [ ]:
X = df.drop(columns=["Survived"])   # inputs: Age, Pclass, Fare
y = df["Survived"]                   # target

# split first: the imputer may only learn from the training rows
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.head()

## 3. KNN imputation, then logistic regression

In [ ]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# lbfgs warns that it needs more iterations on the unscaled Fare column; the result is the same
warnings.filterwarnings("ignore")

knn = KNNImputer(n_neighbors=3, weights="distance")
X_train_trf = knn.fit_transform(X_train)   # fit: remember the training rows; transform: fill their gaps
X_test_trf = knn.transform(X_test)         # test gaps are filled from training neighbours only

lr = LogisticRegression()
lr.fit(X_train_trf, y_train)
accuracy_score(y_test, lr.predict(X_test_trf))

## 4. Compare with mean imputation

In [ ]:
from sklearn.impute import SimpleImputer

si = SimpleImputer()                       # strategy="mean"
X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

lr = LogisticRegression()
lr.fit(X_train_trf2, y_train)
accuracy_score(y_test, lr.predict(X_test_trf2))

## 5. Try every k and both weightings

In [ ]:
def accuracy(imputer):
    # fit the imputer and the model on the training set, score on the test set
    model = LogisticRegression().fit(imputer.fit_transform(X_train), y_train)
    return accuracy_score(y_test, model.predict(imputer.transform(X_test)))

results = pd.DataFrame({w: [accuracy(KNNImputer(n_neighbors=k, weights=w)) for k in range(1, 11)]
                        for w in ["uniform", "distance"]}, index=range(1, 11))
results.index.name = "k"
results.round(3)

In [ ]:
# one test passenger is worth 1/179 of accuracy
print(len(X_test), round(1 / len(X_test), 4))

## 6. Adding a missing indicator

In [ ]:
knn_ind = KNNImputer(n_neighbors=3, weights="distance", add_indicator=True)
out = knn_ind.fit_transform(X_train)
print(knn_ind.get_feature_names_out())   # the extra column marks the rows whose Age was missing
print(accuracy(knn_ind))